[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C03_LLM_Evals_Course/06_elicitation/06_elicitation_passk.ipynb)

# 模块 06 · 能力引出与 pass@k —— 动手实验

配套讲解：`06_讲解.html` ｜ 默认 **CPU 即可跑全部内容**（实验三的真实模型分支可选）

本 notebook 用三组实验 + 三道练习，把讲解里的核心论断变成可复现的代码：

| 实验 | 内容 | 对应讲解 |
|---|---|---|
| 实验一（纯模拟） | 每题真实成功率 $p_i \sim \mathrm{Beta}$，蒙特卡洛对比三种 pass@k 估计量的偏差：**聚合式朴素估计系统性高估、逐题插入式低估、Chen 2021 估计量无偏** [Chen 2021] | §3 |
| 实验二（纯模拟） | benchmark 级 pass@k 的 **bootstrap 置信区间**（模块 02 思想复用） | §4 |
| 实验三（小模型可选 / mock 回退） | 同一模型 base vs **CoT 模板**的 pass@1 / pass@4 与 **BoN 增长曲线** —— elicitation gap 的最小演示 [Wei 2022; METR 2024] | §5–§7 |

之后是 ✏️ 练习 1–3（自带自测 assert）与文末 📖 参考答案。

依赖：`numpy`、`matplotlib`；实验三若启用真实模型需 `transformers`、`torch`（默认关闭，用统计结构等价的 mock 模拟器）。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import comb, isclose

rng = np.random.default_rng(0)

def pass_at_k(n, c, k):
    # Chen et al. 2021 无偏估计量  1 - C(n-c, k) / C(n, k)
    # 数值稳定的逐项乘积形式：C(n-c,k)/C(n,k) = prod_{i=n-c+1}^{n} (1 - k/i)
    # 每个因子都在 [0,1] 内，n 再大也不会溢出。
    if k > n:
        raise ValueError("k 不能大于 n")
    if n - c < k:          # 失败样本不足 k 个 -> 任取 k 个必含正确解
        return 1.0
    return float(1.0 - np.prod(1.0 - k / np.arange(n - c + 1, n + 1)))

# 自检：与精确组合数对照
assert isclose(pass_at_k(10, 3, 5), 1 - comb(7, 5) / comb(10, 5))   # = 11/12
assert pass_at_k(10, 0, 5) == 0.0                                   # c=0：空乘积=1 -> 0
assert pass_at_k(10, 10, 3) == 1.0                                  # c=n -> 1
assert isclose(pass_at_k(8, 2, 1), 2 / 8)                           # k=1 退化为 c/n
print("pass_at_k 自检通过：pass@5(n=10, c=3) =", round(pass_at_k(10, 3, 5), 6))


## 实验一：三种 pass@k 估计量的偏差（蒙特卡洛）

设定：$M=200$ 道题，每题真实 per-sample 成功率 $p_i \sim \mathrm{Beta}(0.5,\,2.0)$（均值 0.2、重尾偏向 0 —— 刻意制造**难度异质**），每题独立采样 $n=50$ 次得到通过数 $c_i \sim \mathrm{Bin}(n, p_i)$。

真值与三个估计量（$f(p)=1-(1-p)^k$ 对 $k\ge2$ 严格凹）：

- **真值**：$\text{Pass@}k = \frac1M\sum_i \left[1-(1-p_i)^k\right]$
- **朴素 A（聚合式）**：$1-(1-\bar p)^k$，$\bar p=\sum_i c_i/(Mn)$ 是全 benchmark 汇总的单一成功率。Jensen（**题间**）$\Rightarrow$ **系统性高估**，且偏差不随 $n\to\infty$ 消失——这是协议错误，不是噪声。
- **朴素 B（逐题插入式）**：$\frac1M\sum_i\left[1-(1-c_i/n)^k\right]$。Jensen（**题内**采样噪声）$\Rightarrow$ **低估**；Chen et al. 2021 附录指出该偏差在 $n>5k$ 时仍未完全消失 [Chen 2021]。
- **无偏（Chen 2021）**：$\frac1M\sum_i\left[1-\binom{n-c_i}{k}/\binom{n}{k}\right]$，对任意 $n\ge k$ 严格无偏（推导见讲解 §3.3）。

下面重复 $R=300$ 次"生成 $c_i$ → 计算三个估计"的实验，画出偏差（估计的均值 − 真值）随 $k$ 的曲线。


In [ ]:
M, N = 200, 50
p_true = rng.beta(0.5, 2.0, size=M)        # 每题真实 per-sample 成功率（评测中不可观测）
ks = np.array([1, 2, 4, 8, 16, 32, 50])
true_passk = np.array([float((1 - (1 - p_true) ** int(k)).mean()) for k in ks])

R = 300                                     # 蒙特卡洛重复次数（约数秒）
est = {name: np.zeros((R, len(ks))) for name in ("pooled", "plugin", "unbiased")}
for r in range(R):
    c = rng.binomial(N, p_true)             # 一次"评测"：每题 n=50 采样的通过数
    p_bar = c.sum() / (M * N)               # 聚合式朴素估计用的全局成功率
    for j, k_ in enumerate(ks):
        k_ = int(k_)
        est["pooled"][r, j] = 1 - (1 - p_bar) ** k_
        est["plugin"][r, j] = float((1 - (1 - c / N) ** k_).mean())
        est["unbiased"][r, j] = float(np.mean([pass_at_k(N, int(ci), k_) for ci in c]))

labels = {"pooled":   "naive A: pooled p-hat (overestimates)",
          "plugin":   "naive B: per-problem plug-in (underestimates)",
          "unbiased": "Chen 2021 unbiased"}
plt.figure(figsize=(7, 4.2))
for name, marker in (("pooled", "^"), ("plugin", "v"), ("unbiased", "o")):
    plt.plot(ks, est[name].mean(axis=0) - true_passk, marker=marker, label=labels[name])
plt.axhline(0, color="gray", lw=1)
plt.xscale("log"); plt.xticks(ks, ks)
plt.xlabel("k"); plt.ylabel("bias = E[estimate] - truth")
plt.title(f"Bias of pass@k estimators (M={M}, n={N}, {R} MC reps)")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

print(f"{'k':>4}{'truth':>9}{'pooled':>9}{'plugin':>9}{'unbiased':>10}")
for j, k_ in enumerate(ks):
    print(f"{int(k_):>4}{true_passk[j]:>9.4f}{est['pooled'][:, j].mean():>9.4f}"
          f"{est['plugin'][:, j].mean():>9.4f}{est['unbiased'][:, j].mean():>10.4f}")
print("\n读图：聚合式在 k 大时高估 20+ 个点且不随 n 消失；插入式随 k->n 越压越低；无偏估计量贴着 0。")


In [ ]:
# 实验二：benchmark 级 Pass@k 的 bootstrap CI —— 对"题目"维度重采样（模块 02 思想）
# 先把每题的无偏 pass@k 算成定值 s_i，再对 {s_i} 重采样取均值。
# 注意：这把"生成随机性"折叠进了每题点估计；更完整的做法是题目 x 样本两层分层重采样（见模块 02）。
c_obs = rng.binomial(N, p_true)             # 一次真实评测观测到的每题通过数
B = 2000
for k_ in (1, 10, 50):
    s = np.array([pass_at_k(N, int(ci), k_) for ci in c_obs])     # 每题的无偏 pass@k
    point = float(s.mean())
    boots = np.array([s[rng.integers(0, M, M)].mean() for _ in range(B)])
    lo, hi = np.percentile(boots, [2.5, 97.5])
    print(f"Pass@{k_:<3} = {point:.3f}   95% CI [{lo:.3f}, {hi:.3f}]   宽度 {hi - lo:.3f}")
print("\n结论（讲解 §4）：报告 pass@k 必须连同 n、温度与 CI 一起报；"
      "k 接近 n 时逐题估计退化为指示函数 1{c>=1}，方差不再被 n 平均掉。")


## 实验三：CoT 引出增益与 BoN 曲线（真实小模型可选，默认 mock）

5 道**程序可验证**的算术 / 字符串谜题，每题在 temperature = 0.8 下采样 $k=8$ 次（高 $k$ 的 pass@k 用更高温度，是 HumanEval 的实践 [Chen 2021]）。每题跑两套 harness：

- **base**：直接要答案（引出阶梯 L0）；
- **CoT**：要求一步一步推理、最后一行给 `答案：<x>`（引出阶梯 L1 [Wei 2022]）。

判分器是**精确匹配**（oracle verifier），因此 **BoN@n ≡ pass@n**（讲解 §5），可以直接画 BoN 分数随 $n$ 的增长曲线；base 与 CoT 的分差就是这一级的 **elicitation gap**（讲解 §7；METR 视角：分数永远是引出水平的函数 [METR 2024]）。

**两种运行方式**：

- `USE_REAL_MODEL = False`（默认）：用 `MockLM` —— 每题带 base / CoT 两档"真实"成功率，按 $\mathrm{Bernoulli}(p)$ 返回对 / 错答案。与真实采样**统计结构同构**，CPU 秒级跑完，全部指标照常演示。
- `USE_REAL_MODEL = True`：用 `Qwen/Qwen2.5-1.5B-Instruct`（约 3.1 GB 下载；CPU 可跑但 5 题 × 2 条件 × 8 次采样约需十几分钟，GPU/MPS 数分钟）。

> mock 任务里的 `p_base` / `p_cot` 字段只是模拟器的真值，真实模型分支会忽略它们。


In [ ]:
USE_REAL_MODEL = False   # <- 改 True 使用 Qwen/Qwen2.5-1.5B-Instruct（约 3.1GB 下载）

TASKS = [
    dict(q="计算 27 * 14。",                       answer="378",    wrong=["368", "388", "278"],      p_base=0.55, p_cot=0.90),
    dict(q="把字符串 'metric' 反转。",              answer="cirtem", wrong=["metric", "cirtme", "mirtec"], p_base=0.25, p_cot=0.60),
    dict(q="计算 139 + 286。",                     answer="425",    wrong=["415", "435", "424"],      p_base=0.60, p_cot=0.95),
    dict(q="单词 'elicitation' 一共有多少个字母？",  answer="11",     wrong=["10", "12", "13"],         p_base=0.35, p_cot=0.70),
    dict(q="计算 7 的 4 次方。",                    answer="2401",   wrong=["16384", "343", "2801"],   p_base=0.30, p_cot=0.75),
]

def extract_answer(text):
    # 取最后一个非空行，剥掉常见前缀与标点，统一小写（最小可用的答案抽取，模块 03 详述其陷阱）
    lines = [l.strip() for l in text.strip().splitlines() if l.strip()]
    last = lines[-1] if lines else ""
    for prefix in ("答案：", "答案:", "Answer:", "answer:"):
        if last.startswith(prefix):
            last = last[len(prefix):]
    return last.strip(" 。．.!！`*").lower()

def make_verifier(task):
    # oracle verifier：答案精确匹配（程序可验证域）
    gold = task["answer"].lower()
    def verify(output):
        return extract_answer(output) == gold
    return verify

class MockLM:
    # 确定性模拟器：每次"采样" ~ Bernoulli(p)，p 取该题 base/CoT 档的真实成功率。
    # 与真实模型采样在统计结构上同构，足以演示 pass@k / BoN / elicitation gap。
    def __init__(self, seed=42):
        self.rng = np.random.default_rng(seed)
    def generate(self, task, cot, temperature=0.8):
        p = task["p_cot"] if cot else task["p_base"]
        ok = self.rng.random() < p
        ans = task["answer"] if ok else task["wrong"][int(self.rng.integers(len(task["wrong"])))]
        head = "让我一步一步推理……（模拟的推理轨迹）\n" if cot else ""
        return head + "答案：" + ans

if USE_REAL_MODEL:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
    tok = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype="auto", device_map="auto")
    def sample_once(task, cot, temperature=0.8):
        suffix = ("\n请一步一步推理，最后一行以「答案：<最终答案>」的格式单独给出最终答案。" if cot
                  else "\n直接给出最终答案，不要解释。格式：答案：<最终答案>")
        msgs = [{"role": "user", "content": task["q"] + suffix}]
        text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        ids = tok(text, return_tensors="pt").to(model.device)
        out = model.generate(**ids, do_sample=True, temperature=temperature, top_p=0.95,
                             max_new_tokens=256, pad_token_id=tok.eos_token_id)
        return tok.decode(out[0, ids["input_ids"].shape[1]:], skip_special_tokens=True)
else:
    _mock = MockLM(seed=42)
    def sample_once(task, cot, temperature=0.8):
        return _mock.generate(task, cot, temperature)

K_SAMPLES = 8
results = {}                                   # (题号, 条件) -> {"outputs": [...], "c": 通过数}
for ti, task in enumerate(TASKS):
    verify = make_verifier(task)
    for cond in ("base", "cot"):
        outs = [sample_once(task, cot=(cond == "cot")) for _ in range(K_SAMPLES)]
        results[(ti, cond)] = {"outputs": outs, "c": int(sum(verify(o) for o in outs))}
print(f"采样完成：{len(TASKS)} 题 x 2 条件 x {K_SAMPLES} 次（temperature=0.8）")
print("示例输出（题 1, CoT 条件）：", repr(results[(0, 'cot')]['outputs'][0]))


In [ ]:
# --- pass@1 / pass@4（n=8 的无偏估计）与 CoT 引出增益 ---
p1 = {"base": [], "cot": []}
p4 = {"base": [], "cot": []}
print(f"{'题':<3}{'c_base':>7}{'c_cot':>6} | {'p@1 base':>9}{'p@1 cot':>8} | {'p@4 base':>9}{'p@4 cot':>8}")
for ti in range(len(TASKS)):
    cb, cc = results[(ti, "base")]["c"], results[(ti, "cot")]["c"]
    for cond, c in (("base", cb), ("cot", cc)):
        p1[cond].append(pass_at_k(K_SAMPLES, c, 1))
        p4[cond].append(pass_at_k(K_SAMPLES, c, 4))
    print(f"{ti + 1:<3}{cb:>7}{cc:>6} | {p1['base'][-1]:>9.3f}{p1['cot'][-1]:>8.3f}"
          f" | {p4['base'][-1]:>9.3f}{p4['cot'][-1]:>8.3f}")

gap1 = float(np.mean(p1["cot"]) - np.mean(p1["base"]))
gap4 = float(np.mean(p4["cot"]) - np.mean(p4["base"]))
print(f"\n均值  pass@1: base={np.mean(p1['base']):.3f}  cot={np.mean(p1['cot']):.3f}  -> CoT 增益 {gap1:+.3f}")
print(f"均值  pass@4: base={np.mean(p4['base']):.3f}  cot={np.mean(p4['cot']):.3f}  -> CoT 增益 {gap4:+.3f}")
print("这两个增益就是 L0->L1 的 elicitation gap（练习 3 会给它配上配对 bootstrap CI）。")

# --- BoN 曲线：oracle verifier 下 BoN@n == pass@n，画分数随采样预算 n 的增长 ---
ns = np.arange(1, K_SAMPLES + 1)
plt.figure(figsize=(6.4, 4))
for cond, marker in (("base", "o"), ("cot", "s")):
    curve = [float(np.mean([pass_at_k(K_SAMPLES, results[(ti, cond)]["c"], int(n_))
                            for ti in range(len(TASKS))])) for n_ in ns]
    plt.plot(ns, curve, marker=marker, label=cond)
plt.xlabel("n (sampling budget)"); plt.ylabel("BoN@n = pass@n  (oracle verifier)")
plt.title("Best-of-n score vs sampling budget (capability probe)")
plt.ylim(0, 1.02); plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
print("读图：曲线随 n 上升 = 多采样在买能力分布的尾部；CoT 曲线整体上移 = 引出水平的提升；"
      "两条曲线的纵向距离随 n 变化 -> 单点比较（只看 pass@1）可能误导。")


## ✏️ 练习 1：无偏 pass@k 估计量（数值稳定版）

不回看上文实现，自己写 `pass_at_k_student(n, c, k)`：返回 Chen et al. 2021 的无偏估计
$1 - \binom{n-c}{k}\big/\binom{n}{k}$。

要求：

- **数值稳定**：不要直接算大组合数的比值（$n$ 大时溢出 / 精度丢失），用逐项乘积
  $\prod_{i=n-c+1}^{n}\left(1-\frac{k}{i}\right)$ 或对数累加；
- 边界：`c == 0` → `0.0`；`n - c < k`（含 `c == n`）→ `1.0`；`k > n` → `raise ValueError`。

提示：`np.arange(n - c + 1, n + 1)` 给出乘积的分母序列（共 `c` 项）；空数组的 `np.prod` 等于 1.0，恰好让 `c == 0` 的边界自动成立。10 行以内可完成。


In [ ]:
def pass_at_k_student(n: int, c: int, k: int) -> float:
    # 返回无偏估计 1 - C(n-c, k) / C(n, k)，要求数值稳定
    # TODO: 1) k > n 时 raise ValueError
    # TODO: 2) n - c < k 时返回 1.0
    # TODO: 3) 其余情况用逐项乘积 prod_{i=n-c+1}^{n} (1 - k/i) 计算
    raise NotImplementedError


In [ ]:
# ---- 练习 1 自测 ----
from math import comb, isclose

assert isclose(pass_at_k_student(10, 3, 5), 1 - comb(7, 5) / comb(10, 5), rel_tol=1e-9)  # = 11/12
assert pass_at_k_student(10, 0, 5) == 0.0          # c = 0
assert pass_at_k_student(10, 10, 3) == 1.0         # c = n
assert pass_at_k_student(10, 8, 5) == 1.0          # k > n - c
assert pass_at_k_student(50, 1, 50) == 1.0         # k = n 且 c >= 1

# 与精确组合数随机对照（comb(a,k) 在 a<k 时为 0，恰好覆盖边界）
rng_t = np.random.default_rng(1)
for _ in range(200):
    n_ = int(rng_t.integers(1, 60)); k_ = int(rng_t.integers(1, n_ + 1)); c_ = int(rng_t.integers(0, n_ + 1))
    exact = 1 - comb(n_ - c_, k_) / comb(n_, k_)
    assert isclose(pass_at_k_student(n_, c_, k_), exact, rel_tol=1e-9, abs_tol=1e-12), (n_, c_, k_)

# 大 n 不溢出（直接算 C(99950,100)/C(100000,100) 的浮点版会溢出）
v = pass_at_k_student(100000, 50, 100)
assert 0.0 < v < 1.0

try:
    pass_at_k_student(5, 2, 6)
    raise AssertionError("k > n 应当 raise ValueError")
except ValueError:
    pass
print("✅ 练习 1 通过")


## ✏️ 练习 2：`bon_score(samples, verifier)` —— best-of-n 选择

实现 BoN 选择器。`verifier(sample) -> 分数`（数值越大越好；布尔判分器返回 `True/False` 也要兼容——cast 成 float 即可），从候选列表 `samples` 中选出得分最高的样本。

要求：

- 返回二元组 `(best_sample, best_score)`，其中 `best_score` 是 `float`；
- **平分时返回最早出现的样本**（评测代码必须确定性、可复现）；
- `samples` 为空 → `raise ValueError`。

提示：先算 `scores = [float(verifier(s)) for s in samples]`；`np.argmax` 平分时恰好返回第一个最大值的下标。5–8 行可完成。


In [ ]:
def bon_score(samples, verifier):
    # TODO: 1) samples 为空 -> raise ValueError
    # TODO: 2) 对每个 sample 调 verifier 打分（cast 成 float）
    # TODO: 3) 返回 (得分最高且最早出现的 sample, 其得分)
    raise NotImplementedError


In [ ]:
# ---- 练习 2 自测 ----
v_exact = lambda s: s.strip() == "378"                      # 布尔 oracle verifier（精确匹配）
assert bon_score(["377", "378", "999"], v_exact) == ("378", 1.0)
assert bon_score(["1", "2"], v_exact) == ("1", 0.0)         # 全错 -> 平分取最早

v_len = lambda s: float(len(s))                             # 数值 verifier
assert bon_score(["ab", "abcd", "abc"], v_len) == ("abcd", 4.0)
assert bon_score(["x", "yy", "zz"], v_len) == ("yy", 2.0)   # 平分取最早

# 与本模块管线对接：oracle verifier 下 BoN 选中的就是任一正确样本
task0 = TASKS[0]
v0 = make_verifier(task0)
best, score = bon_score(["答案：999", "答案：378", "答案：368"], v0)
assert best == "答案：378" and score == 1.0

try:
    bon_score([], v_exact)
    raise AssertionError("空列表应当 raise ValueError")
except ValueError:
    pass
print("✅ 练习 2 通过")


## ✏️ 练习 3：`elicitation_gap` —— 逐题 gap 与配对 bootstrap CI

把实验三里的"CoT 增益"升级成带不确定性度量的标准工具（模块 02 的**配对 bootstrap** 思想）。

输入同一批题在两套 harness 下的逐题分数 `scores_base`、`scores_scaffold`（等长、按题配对，例如每题的 pass@1），实现
`elicitation_gap(scores_base, scores_scaffold, n_boot=2000, seed=0)`，返回 `(gaps, (lo, hi))`：

1. `gaps`：逐题 gap 的 `np.ndarray`，定义为 `scaffold - base`；
2. `(lo, hi)`：平均 gap 的 95% **配对** bootstrap CI——重采样**题目下标**（同一下标同时取两边，保持配对结构），每次算 `gaps[idx].mean()`，最后取 2.5 / 97.5 百分位。

要求：两组分数长度不一致 → `raise ValueError`；用 `np.random.default_rng(seed)` 保证可复现。

提示：配对设计消掉了"题目难度"这一最大方差来源——对 `gaps` 重采样即可，不需要分别对两组重采样。10–15 行可完成。


In [ ]:
def elicitation_gap(scores_base, scores_scaffold, n_boot=2000, seed=0):
    # TODO: 1) 转成 float np.array，长度不一致 -> raise ValueError
    # TODO: 2) gaps = scaffold - base（逐题）
    # TODO: 3) rng_local = np.random.default_rng(seed)；重复 n_boot 次：
    #          抽 m 个题目下标（有放回）-> 记录 gaps[idx].mean()
    # TODO: 4) lo, hi = 2.5 / 97.5 百分位；返回 (gaps, (float(lo), float(hi)))
    raise NotImplementedError


In [ ]:
# ---- 练习 3 自测 ----
base_s = np.array([0.2, 0.3, 0.1, 0.4, 0.2, 0.3, 0.1, 0.5, 0.3, 0.2])
scaf_s = base_s + 0.3
gaps, (lo, hi) = elicitation_gap(base_s, scaf_s, n_boot=500, seed=0)
assert gaps.shape == (10,) and np.allclose(gaps, 0.3)
assert abs(lo - 0.3) < 1e-9 and abs(hi - 0.3) < 1e-9       # 所有 gap 相同 -> CI 退化为一个点

base2 = np.linspace(0.1, 0.4, 40)
g_design = np.linspace(0.05, 0.35, 40)                      # 平均 gap = 0.2，逐题有差异
gaps2, (lo2, hi2) = elicitation_gap(base2, base2 + g_design, seed=1)
assert np.allclose(gaps2, g_design)
assert 0 < lo2 < 0.2 < hi2                                  # 方向正确：scaffold 显著更好，CI 下界 > 0

flat = np.concatenate([np.full(20, 0.05), np.full(20, -0.05)])   # 平均 gap 恰为 0
gaps3, (lo3, hi3) = elicitation_gap(base2, base2 + flat, seed=2)
assert lo3 < 0 < hi3                                        # 无真实提升 -> CI 跨 0

try:
    elicitation_gap(np.ones(3), np.ones(4))
    raise AssertionError("长度不一致应当 raise ValueError")
except ValueError:
    pass

# 用在实验三的真实产出上：CoT 对 pass@1 的逐题增益 + CI
g_cot, (l_cot, h_cot) = elicitation_gap(p1["base"], p1["cot"], seed=3)
print(f"CoT elicitation gap (pass@1)：均值 {g_cot.mean():+.3f}，95% CI [{l_cot:+.3f}, {h_cot:+.3f}]（仅 5 题，CI 很宽是正常的）")
print("✅ 练习 3 通过")


## 📖 参考答案

> **先自己做，再对照。**三段参考实现都能让上面的自测 cell 全部通过；如果你的实现思路不同但 assert 全过，同样算完成。


In [ ]:
# 参考答案 1（先自己做，再对照）
def pass_at_k_student(n: int, c: int, k: int) -> float:
    if k > n:
        raise ValueError("k 不能大于 n")
    if n - c < k:                  # 含 c == n；失败样本不足 k 个，任取 k 个必含正确解
        return 1.0
    # C(n-c,k)/C(n,k) = prod_{i=n-c+1}^{n} (i-k)/i，每个因子在 [0,1] 内 -> 数值稳定
    return float(1.0 - np.prod(1.0 - k / np.arange(n - c + 1, n + 1)))


In [ ]:
# 参考答案 2（先自己做，再对照）
def bon_score(samples, verifier):
    if len(samples) == 0:
        raise ValueError("samples 不能为空")
    scores = [float(verifier(s)) for s in samples]
    i = int(np.argmax(scores))     # np.argmax 平分时返回最早的下标 -> 确定性
    return samples[i], scores[i]


In [ ]:
# 参考答案 3（先自己做，再对照）
def elicitation_gap(scores_base, scores_scaffold, n_boot=2000, seed=0):
    b = np.asarray(scores_base, dtype=float)
    s = np.asarray(scores_scaffold, dtype=float)
    if b.shape != s.shape:
        raise ValueError("两组分数必须等长（逐题配对）")
    gaps = s - b
    rng_local = np.random.default_rng(seed)
    m = len(gaps)
    boots = np.array([gaps[rng_local.integers(0, m, m)].mean() for _ in range(n_boot)])
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return gaps, (float(lo), float(hi))


## 小结

- **估计量要选对**：聚合式朴素估计因题目难度异质而**系统性高估**（协议错误，不随 $n$ 消失）；逐题插入式因题内采样噪声而**低估**（$n>5k$ 仍可见）；Chen 2021 的 $1-\binom{n-c}{k}/\binom{n}{k}$ 对任意 $n\ge k$ 无偏，且必须用逐项乘积实现 [Chen 2021]。
- **不确定性要报**：pass@k 必须连同 $n$、温度与 bootstrap CI 一起报告；$k\to n$ 时估计退化、方差激增。
- **分数是引出水平的函数**：同一模型换一个 CoT 模板，pass@1 / pass@4 显著上移 [Wei 2022]；oracle verifier 下 BoN@n ≡ pass@n，是最便宜的能力上界探针；elicitation gap 要用配对 bootstrap 给 CI。
- **安全评测的含义**：under-elicitation 产生方向性的假阴性（虚假安心）——任何"模型不具备能力 X"的结论都必须附带引出水平声明 [METR 2024]。

**下一步 → 模块 07 · 从零搭建 eval harness**：把本模块的采样、判分、pass@k、引出配置全部工程化成可复现的 harness——你会发现"引出水平"恰好就是 harness 的配置项。


---
## 🎯 真实数据胶囊题：真实任务上的无偏 pass@k 估计

pass@k = 采 k 次至少对一次的概率。直接采 k 次估计方差大；用**无偏估计量**（采 n 次中有 c 次对，pass@k = 1 - C(n-c,k)/C(n,k)）。在真实 GSM8K 难度结构上验证。

> 本模块新增的**真实数据**练习：自包含、用真实公开数据把本章方法跑一遍。先做 TODO，`assert` 全过即通关，文末有参考答案。

In [ ]:
import os, json, urllib.request, re
import numpy as np
CACHE=os.path.expanduser("~/.llm_evals_data"); os.makedirs(CACHE,exist_ok=True)
def _f(url,fn):
    p=os.path.join(CACHE,fn)
    if not os.path.exists(p): urllib.request.urlretrieve(url,p)
    return p
def gsm8k(n=300):
    p=_f("https://raw.githubusercontent.com/openai/grade-school-math/master/grade_school_math/data/test.jsonl","gsm8k_test.jsonl")
    return [json.loads(l) for l in open(p).read().splitlines()[:n]]
def gold(ans): return ans.split("####")[-1].strip().replace(",","")
def shakespeare():
    return open(_f("https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt","shake.txt")).read()

rows=gsm8k(200)
def steps(a): return a.count("<<")
diffs=np.array([steps(r["answer"]) for r in rows])
rng=np.random.default_rng(0)
# 每题采 n 次，单次正确率随真实难度下降
n_samp=20
p=1/(1+np.exp((diffs-diffs.mean())/2.5))
samples=(rng.random((len(rows),n_samp)) < p[:,None]).astype(int)
c=samples.sum(1)  # 每题 n 次里对了几次
print(f"每题采 {n_samp} 次, 平均 pass@1={ (c/n_samp).mean():.3f}")

**练习**：实现 `pass_at_k(n, c, k)`（单题，无偏估计 `1 - C(n-c,k)/C(n,k)`，c<k 时若 n-c<k 则为1处理边界）。

In [ ]:
def pass_at_k(n, c, k):
    # TODO: 若 n-c < k 返回 1.0；否则 1 - comb(n-c,k)/comb(n,k)
    raise NotImplementedError


In [ ]:
# 自测
from math import comb
assert pass_at_k(20,0,5)==0.0, "0次对 -> pass@k=0"
assert pass_at_k(20,20,5)==1.0, "全对 -> pass@k=1"
assert abs(pass_at_k(10,1,1)-0.1)<1e-9, "对1次 pass@1=1/10"
# pass@k 随 k 单调不减
p1=np.mean([pass_at_k(n_samp,ci,1) for ci in c])
p5=np.mean([pass_at_k(n_samp,ci,5) for ci in c])
assert p5 >= p1, "pass@5 >= pass@1"
print(f"真实数据: pass@1={p1:.3f} -> pass@5={p5:.3f} ✓ (k越大越高)")


### 📖 参考答案

In [ ]:
from math import comb
def pass_at_k(n, c, k):
    if n-c < k: return 1.0
    return 1.0 - comb(n-c,k)/comb(n,k)
print("✓ 无偏 pass@k (Codex 论文) 避免了小样本高方差")